# TruthLens — Phase 2: DistilBERT Model Training on Google Colab
### An NLP-Based Fake News Detection and Robustness Evaluation System
**Objective:** Fine-tune `distilbert-base-uncased` on the LIAR dataset for 6-class claim credibility classification using Google Colab GPU.

---
### Runtime Requirement:
Please ensure your Colab runtime is set to **GPU** before running:
- **Runtime** -> **Change runtime type** -> **Hardware accelerator: T4 GPU** (or available GPU).


## Section 1: Environment Setup
- Install required packages
- Import libraries
- Fix random seeds for reproducibility
- Detect CUDA availability & GPU hardware specifications


In [ ]:
# Install required packages
!pip install -q torch transformers datasets scikit-learn pandas numpy matplotlib seaborn huggingface_hub

import os
import sys
import csv
import json
import time
import random
import urllib.request
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# Set random seeds for exact reproducibility
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Verify PyTorch & GPU Hardware
print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"SUCCESS: GPU detected: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    print("WARNING: GPU is NOT available. Running on CPU. Training will be slow!")
    print("Please navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")


## Section 2: Dataset Acquisition
Download the verified benchmark LIAR dataset splits (`train.tsv`, `valid.tsv`, `test.tsv`), validate column schema, and inspect the 6-class distributions.


In [ ]:
# Setup dataset directories
DATA_DIR = "datasets/liar"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/"
SPLITS = ["train.tsv", "valid.tsv", "test.tsv"]
headers = {"User-Agent": "Mozilla/5.0"}

for split in SPLITS:
    target_path = os.path.join(DATA_DIR, split)
    if not os.path.exists(target_path):
        url = BASE_URL + split
        print(f"Downloading {split} from {url}...")
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req) as resp, open(target_path, "wb") as out_f:
            out_f.write(resp.read())

print("Dataset files validated locally:")
for split in SPLITS:
    size = os.path.getsize(os.path.join(DATA_DIR, split))
    print(f" - {split}: {size / 1024:.1f} KB")


In [ ]:
# Define the canonical 6-class truthfulness labels (in increasing order of veracity)
LABEL_LIST = [
    "pants-fire",
    "false",
    "barely-true",
    "half-true",
    "mostly-true",
    "true"
]

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

def load_liar_split(file_path):
    samples = []
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if len(row) >= 3:
                sample_id = row[0].strip()
                label = row[1].strip().lower()
                statement = row[2].strip()
                speaker = row[4].strip() if len(row) > 4 else ""
                context = row[13].strip() if len(row) > 13 else ""
                if label in LABEL2ID and statement:
                    samples.append({
                        "id": sample_id,
                        "label": label,
                        "label_id": LABEL2ID[label],
                        "statement": statement,
                        "speaker": speaker,
                        "context": context
                    })
    return samples

train_data = load_liar_split(os.path.join(DATA_DIR, "train.tsv"))
valid_data = load_liar_split(os.path.join(DATA_DIR, "valid.tsv"))
test_data = load_liar_split(os.path.join(DATA_DIR, "test.tsv"))

print("Dataset Split Sizes:")
print(f"  Train samples:      {len(train_data):,}")
print(f"  Validation samples: {len(valid_data):,}")
print(f"  Test samples:       {len(test_data):,}")
print(f"  Total samples:      {len(train_data) + len(valid_data) + len(test_data):,}")

# Plot training class distribution
train_df = pd.DataFrame(train_data)
dist = train_df["label"].value_counts()[LABEL_LIST]
print("\nTraining Set Label Counts:")
print(dist)

plt.figure(figsize=(9, 4))
sns.barplot(x=dist.index, y=dist.values, palette="Blues_r")
plt.title("LIAR Training Set — 6-Class Truthfulness Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Veracity Class")
plt.ylabel("Statement Count")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## Section 3: Preprocessing & PyTorch Dataset Preparation
Tokenize statements using Hugging Face `DistilBertTokenizerFast`.
- Maximum Sequence Length: 128 tokens
- Preserves natural statement semantics (no aggressive NLP stopword removal or stemming)
- PyTorch DataLoaders configured with batch size 16


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16

print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LiarTorchDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_len=128):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer(
            item["statement"],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(item["label_id"], dtype=torch.long)
        }

train_dataset = LiarTorchDataset(train_data, tokenizer, max_len=MAX_LENGTH)
valid_dataset = LiarTorchDataset(valid_data, tokenizer, max_len=MAX_LENGTH)
test_dataset = LiarTorchDataset(test_data, tokenizer, max_len=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created:")
print(f"  Train batches: {len(train_loader)} (Batch size {BATCH_SIZE})")
print(f"  Valid batches: {len(valid_loader)}")
print(f"  Test batches:  {len(test_loader)}")


## Section 4: Baseline Model (TF-IDF + Logistic Regression)
Train a TF-IDF + Logistic Regression baseline model on the original training split and evaluate it on the unperturbed test split to provide an empirical point of comparison.


In [ ]:
print("Training Baseline: TF-IDF (1,2 n-grams) + Logistic Regression...")
X_train_text = [item["statement"] for item in train_data]
y_train_labels = [item["label_id"] for item in train_data]

X_test_text = [item["statement"] for item in test_data]
y_test_labels = [item["label_id"] for item in test_data]

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), lowercase=True)
X_train_vec = tfidf.fit_transform(X_train_text)
X_test_vec = tfidf.transform(X_test_text)

log_reg = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
log_reg.fit(X_train_vec, y_train_labels)

baseline_preds = log_reg.predict(X_test_vec)

baseline_acc = accuracy_score(y_test_labels, baseline_preds)
baseline_macro_p, baseline_macro_r, baseline_macro_f1, _ = precision_recall_fscore_support(
    y_test_labels, baseline_preds, average="macro", zero_division=0
)
baseline_weighted_p, baseline_weighted_r, baseline_weighted_f1, _ = precision_recall_fscore_support(
    y_test_labels, baseline_preds, average="weighted", zero_division=0
)

print("\n======================================================")
print("  TF-IDF + Logistic Regression Baseline Results")
print("======================================================")
print(f"  Accuracy:        {baseline_acc:.4f}")
print(f"  Macro Precision: {baseline_macro_p:.4f}")
print(f"  Macro Recall:    {baseline_macro_r:.4f}")
print(f"  Macro F1:        {baseline_macro_f1:.4f}")
print(f"  Weighted F1:     {baseline_weighted_f1:.4f}")
print("======================================================")


## Section 5: DistilBERT Model Fine-Tuning
Fine-tune `distilbert-base-uncased` with:
- Learning Rate: `2e-5`
- Batch Size: `16`
- Epochs: `3`
- Weight Decay: `0.01`
- Optimizer: `AdamW`
- Scheduler: Linear with 10% warmup
- Best checkpoint selected via validation Macro F1


In [ ]:
OUTPUT_DIR = "models/distilbert-liar"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LEARNING_RATE = 2e-5
EPOCHS = 3
WEIGHT_DECAY = 0.01

print(f"Initializing DistilBERT sequence classifier ({len(LABEL_LIST)} classes)...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
model.to(device)

no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, p, r, f1, w_f1

print("Starting DistilBERT Training Loop on Colab GPU...")
start_time = time.time()
best_val_macro_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        if step % 150 == 0 or step == len(train_loader):
            print(f"Epoch {epoch}/{EPOCHS} | Batch {step}/{len(train_loader)} | Train Loss: {running_loss/step:.4f}")

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc, val_p, val_r, val_f1, val_wf1 = evaluate(model, valid_loader, device)
    
    print(f"--> Epoch {epoch} Results | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_f1:.4f} | Val Weighted F1: {val_wf1:.4f}")
    
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
        "val_weighted_f1": val_wf1
    })

    if val_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_f1
        print(f"    * New best validation Macro F1 ({best_val_macro_f1:.4f}). Saving model checkpoint to {OUTPUT_DIR}...")
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

total_training_duration = time.time() - start_time
print(f"\nTraining completed in {total_training_duration/60:.2f} minutes.")


## Section 6: Model Evaluation on Test Split
Benchmark the best saved DistilBERT model against the unseen, unperturbed test split.
Calculate Accuracy, Precision, Recall, Macro F1, Weighted F1, Confusion Matrix, and Training Loss Curves.


In [ ]:
print(f"Loading best model from {OUTPUT_DIR} for test set evaluation...")
best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
best_model.to(device)
best_model.eval()

test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(batch["labels"].numpy())

distilbert_acc = accuracy_score(test_targets, test_preds)
distilbert_macro_p, distilbert_macro_r, distilbert_macro_f1, _ = precision_recall_fscore_support(
    test_targets, test_preds, average="macro", zero_division=0
)
distilbert_weighted_p, distilbert_weighted_r, distilbert_weighted_f1, _ = precision_recall_fscore_support(
    test_targets, test_preds, average="weighted", zero_division=0
)

per_class_p, per_class_r, per_class_f1, per_class_supp = precision_recall_fscore_support(
    test_targets, test_preds, average=None, zero_division=0
)

per_class_metrics = {}
for i, label in enumerate(LABEL_LIST):
    per_class_metrics[label] = {
        "precision": round(float(per_class_p[i]), 4),
        "recall": round(float(per_class_r[i]), 4),
        "f1": round(float(per_class_f1[i]), 4),
        "support": int(per_class_supp[i])
    }

print("\n======================================================")
print("  DistilBERT Benchmark Test Results")
print("======================================================")
print(f"  Accuracy:        {distilbert_acc:.4f}")
print(f"  Macro Precision: {distilbert_macro_p:.4f}")
print(f"  Macro Recall:    {distilbert_macro_r:.4f}")
print(f"  Macro F1:        {distilbert_macro_f1:.4f}")
print(f"  Weighted F1:     {distilbert_weighted_f1:.4f}")
print("======================================================\n")

print("Per-Class Metrics Breakdown:")
print(pd.DataFrame(per_class_metrics).T)


In [ ]:
# Visualizations: Confusion Matrix and Learning Curves
cm = confusion_matrix(test_targets, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Confusion Matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, ax=axes[0])
axes[0].set_title("DistilBERT Confusion Matrix (Test Set)", fontweight="bold")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("Ground Truth Label")

# Plot Loss & Macro F1 curves
epochs_range = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
val_f1s = [h["val_macro_f1"] for h in history]

axes[1].plot(epochs_range, train_losses, "o-", label="Train Loss", color="navy")
axes[1].plot(epochs_range, val_losses, "s--", label="Validation Loss", color="crimson")
axes[1].plot(epochs_range, val_f1s, "^-", label="Val Macro F1", color="forestgreen")
axes[1].set_title("Training Dynamics & Validation Macro F1", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Metric Value")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


## Section 7: Baseline vs. DistilBERT Comparison
Compare traditional TF-IDF + Logistic Regression against contextualized DistilBERT on the exact same test split.


In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression (Baseline)",
        "Accuracy": f"{baseline_acc * 100:.2f}%",
        "Macro Precision": f"{baseline_macro_p * 100:.2f}%",
        "Macro Recall": f"{baseline_macro_r * 100:.2f}%",
        "Macro F1": f"{baseline_macro_f1 * 100:.2f}%",
        "Weighted F1": f"{baseline_weighted_f1 * 100:.2f}%"
    },
    {
        "Model": "DistilBERT Fine-Tuned (TruthLens)",
        "Accuracy": f"{distilbert_acc * 100:.2f}%",
        "Macro Precision": f"{distilbert_macro_p * 100:.2f}%",
        "Macro Recall": f"{distilbert_macro_r * 100:.2f}%",
        "Macro F1": f"{distilbert_macro_f1 * 100:.2f}%",
        "Weighted F1": f"{distilbert_weighted_f1 * 100:.2f}%"
    }
])

print("\n--- Comparative Empirical Benchmark Table ---")
print(comparison_df.to_string(index=False))


## Section 8: Model Export & Archival
Save all model weights, tokenizer configuration, label mappings, and evaluation metrics into `models/distilbert-liar/metrics.json` and compress into a ZIP archive for local deployment.


In [ ]:
# Construct final metrics dictionary
metrics_payload = {
    "model_name": "distilbert-base-uncased (Fine-Tuned)",
    "model_identifier": "distilbert-liar",
    "num_classes": len(LABEL_LIST),
    "labels": LABEL_LIST,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "evaluation_date": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "hardware": {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
        "training_duration_seconds": round(total_training_duration, 2)
    },
    "hyperparameters": {
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "max_sequence_length": MAX_LENGTH,
        "weight_decay": WEIGHT_DECAY,
        "optimizer": "AdamW",
        "seed": SEED
    },
    "test_metrics": {
        "accuracy": round(distilbert_acc, 4),
        "macro_precision": round(distilbert_macro_p, 4),
        "macro_recall": round(distilbert_macro_r, 4),
        "macro_f1": round(distilbert_macro_f1, 4),
        "weighted_f1": round(distilbert_weighted_f1, 4)
    },
    "baseline_metrics": {
        "model": "TF-IDF + Logistic Regression",
        "accuracy": round(baseline_acc, 4),
        "macro_f1": round(baseline_macro_f1, 4),
        "weighted_f1": round(baseline_weighted_f1, 4)
    },
    "per_class_metrics": per_class_metrics
}

metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Saved metrics specification to: {metrics_path}")

# Create ZIP archive
ZIP_NAME = "truthlens-distilbert-liar.zip"
!zip -r {ZIP_NAME} {OUTPUT_DIR}
print(f"Archive created: {ZIP_NAME} ({os.path.getsize(ZIP_NAME) / (1024**2):.1f} MB)")

try:
    from google.colab import files
    print("Ready to download trained model archive:")
    files.download(ZIP_NAME)
except Exception as e:
    print(f"Automatic download note: {e}")


## Section 9: Optional Hugging Face Hub Upload
Upload the fine-tuned model and tokenizer directly to Hugging Face Hub.
- Authenticate via Colab Secrets (`HF_TOKEN`) or enter your token securely.
- Tokens and secrets are never logged or hardcoded.


In [ ]:
# Optional Hugging Face Hub Upload
UPLOAD_TO_HF = True  # Set to False to skip upload
HF_REPO_NAME = "truthlens-distilbert-liar"  # Target repository name

if UPLOAD_TO_HF:
    try:
        from huggingface_hub import HfApi, login
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            print("Using HF_TOKEN retrieved from Google Colab Secrets.")
        except Exception:
            import getpass
            hf_token = getpass.getpass("Enter your Hugging Face Write Token: ")

        if hf_token:
            login(token=hf_token, add_to_git_credential=True)
            api = HfApi()
            user_info = api.whoami()
            username = user_info["name"]
            full_repo_id = f"{username}/{HF_REPO_NAME}"
            print(f"Creating/Verifying Hugging Face repository: {full_repo_id}...")
            api.create_repo(repo_id=full_repo_id, exist_ok=True, private=False)

            print(f"Uploading model folder {OUTPUT_DIR} to {full_repo_id}...")
            api.upload_folder(
                folder_path=OUTPUT_DIR,
                repo_id=full_repo_id,
                commit_message="Upload fine-tuned TruthLens DistilBERT model on LIAR dataset"
            )
            print(f"SUCCESS! Model uploaded to: https://huggingface.co/{full_repo_id}")
            print(f"Update your backend .env with: HF_MODEL_REPO={full_repo_id}")
        else:
            print("Skipped: No Hugging Face token provided.")
    except Exception as exc:
        print(f"Hugging Face upload failed: {exc}")
        print("Retaining local ZIP archive for download.")
else:
    print("Hugging Face upload skipped by user setting.")


## Section 10: Final Results Summary & Academic Notes

### Important Dataset & Model Limitations
1. **Short Statement Domain**: The LIAR dataset comprises short political statements, fact-checked debate snippets, and public claims. It is **not** a full-text news article dataset.
2. **Context Scope**: Real-world news verification requires examining long-form narrative structure, citation trails, and external evidence. Performance on LIAR represents an initial 6-class claim-classification baseline.
3. **Fact-Checking Disclaimer**: Model predictions reflect stylistic and lexical patterns learned from training data; they do not constitute definitive, authoritative fact-checking verdicts. Full article analysis and external evidence retrieval will be addressed in Phase 3 and Phase 5.


In [ ]:
print("==================================================================")
print("              TRUTHLENS PHASE 2 TRAINING SUMMARY                  ")
print("==================================================================")
print("Dataset:               LIAR (10,240 train, 1,284 valid, 1,267 test)")
print("Model Architecture:    distilbert-base-uncased")
print("Classes (6):           pants-fire, false, barely-true, half-true, mostly-true, true")
print(f"Hardware:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Duration:              {total_training_duration/60:.2f} minutes")
print(f"Baseline Test Acc:     {baseline_acc:.4f} (Macro F1: {baseline_macro_f1:.4f})")
print(f"DistilBERT Test Acc:   {distilbert_acc:.4f} (Macro F1: {distilbert_macro_f1:.4f})")
print("Exported Artefacts:    models/distilbert-liar/ (metrics.json, model, tokenizer, zip)")
print("==================================================================")
